In [ ]:
import os
import csv
import math as m
chamber = ["right", "left"]
fiber_right_dof = []
fiber_left_dof = []
fiber_right_spring_info = []
fiber_left_spring_info = []

# Specify the path to your CSV file
for i in chamber:
    for j in range(2):
        if i == "right":
            fiber_right_dof.append([]) 
        else:
            fiber_left_dof.append([])
        file_path = os.getcwd()+"/fiber" + str(j+1) + i +"_info.csv"

        # Open the CSV file
        with open(file_path, 'r') as file:
            # Create a CSV reader object as a dictionary reader
            csv_reader = csv.DictReader(file)
            for row in csv_reader:
                if i == "right":
                    fiber_right_dof[-1].append(float(row['x (mm)'])) 
                    fiber_right_dof[-1].append(float(row['y (mm)']))
                    fiber_right_dof[-1].append(float(row['z (mm)']))

                else:
                    fiber_left_dof[-1].append(float(row['x (mm)'])) 
                    fiber_left_dof[-1].append(float(row['y (mm)']))
                    fiber_left_dof[-1].append(float(row['z (mm)']))

 

In [ ]:
fiber_right_dof
print(len(fiber_right_dof))
print(len(fiber_right_dof[0]))

In [ ]:
Ks = 1e3
Kd = 5
for i in range(len(fiber_right_dof)):
    fiber_right_spring_info.append([])
    fiber_left_spring_info.append([])
    for j in range(0,int(len(fiber_right_dof[i])/3)-2):
        fiber_right_spring_info[-1].append([j, j+1, Ks, Kd, m.sqrt((fiber_right_dof[i][3*j]-fiber_right_dof[i][3*(j+1)])**2+
                                                            (fiber_right_dof[i][3*j+1]-fiber_right_dof[i][3*(j+1)+1])**2+
                                                            (fiber_right_dof[i][3*j+2]-fiber_right_dof[i][3*(j+1)+2])**2)])
        
        fiber_left_spring_info[-1].append([j, j+1, Ks, Kd, m.sqrt((fiber_left_dof[i][3*j]-fiber_left_dof[i][3*(j+1)])**2+
                                                            (fiber_left_dof[i][3*j+1]-fiber_left_dof[i][3*(j+1)+1])**2+
                                                            (fiber_left_dof[i][3*j+2]-fiber_left_dof[i][3*(j+1)+2])**2)])

In [ ]:
fiber_dof = [fiber_right_dof, fiber_left_dof]
spring_info = [fiber_right_spring_info, fiber_left_spring_info]

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from PyQt5.QtWidgets import QApplication, QMainWindow, QVBoxLayout, QWidget
from PyQt5.QtCore import Qt, QTimer

class RealTimePlot(QMainWindow):
    def __init__(self):
        super().__init__()

        self.central_widget = QWidget(self)
        self.setCentralWidget(self.central_widget)

        layout = QVBoxLayout(self.central_widget)

        self.figure, self.ax = plt.subplots()
        self.canvas = FigureCanvas(self.figure)
        layout.addWidget(self.canvas)

        self.x = np.linspace(0, 10, 100)
        self.y = np.sin(self.x)

        self.line, = self.ax.plot(self.x, self.y)

        # Set up a timer to update the plot every 100 milliseconds
        self.timer = QTimer(self)
        self.timer.timeout.connect(self.update_plot)
        self.timer.start(100)

    def update_plot(self):
        # Update the data
        self.x += 0.1
        self.y = np.sin(self.x)

        # Update the plot
        self.line.set_data(self.x, self.y)
        self.ax.relim()
        self.ax.autoscale_view()

        # Redraw the canvas
        self.canvas.draw()

if __name__ == '__main__':
    app = QApplication(sys.argv)
    window = RealTimePlot()
    window.show()
    sys.exit(app.exec_())


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
plt.style.use('fivethirtyeight')
import pandas as pd
fieldnames = ["time","xx", "yy", "zz", "yz", "xz", "xy"]
count = 0
def animate(i, count):
    data = pd.read_csv('strain_data.csv')
    no_data = 100
    x = data['time']
    y1 = data['xx']
    y2 = data['yy']
    y3 = data['zz']
    y4 = data['yz']
    y5 = data['xz']
    y6 = data['xy']
    plt.set_xlim(x.min(), x.max())
    plt.cla()

    plt.plot(x[no_data*count:no_data*(count+1)], y1[no_data*count:no_data*(count+1)], label='Channel 1')
    plt.plot(x[no_data*count:no_data*(count+1)], y2[no_data*count:no_data*(count+1)], label='Channel 2')
    plt.plot(x[no_data*count:no_data*(count+1)], y3[no_data*count:no_data*(count+1)], label='Channel 3')
    plt.plot(x[no_data*count:no_data*(count+1)], y4[no_data*count:no_data*(count+1)], label='Channel 4')
    plt.plot(x[no_data*count:no_data*(count+1)], y5[no_data*count:no_data*(count+1)], label='Channel 5')
    plt.plot(x[no_data*count:no_data*(count+1)], y6[no_data*count:no_data*(count+1)], label='Channel 6')

    plt.legend(loc='upper left')
    plt.tight_layout()
    count += 1
    print(count)
    

ani = FuncAnimation(plt.gcf(), animate, frames=10, interval=10)

plt.tight_layout()
plt.show()


In [ ]:
import csv
import random
import time

x_value = 0
total_1 = 1000
total_2 = 1000

fieldnames = ["x_value", "total_1", "total_2"]


with open('data.csv', 'w') as csv_file:
    csv_writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    csv_writer.writeheader()

while True:

    with open('data.csv', 'a') as csv_file:
        csv_writer = csv.DictWriter(csv_file, fieldnames=fieldnames)

        info = {
            "x_value": x_value,
            "total_1": total_1,
            "total_2": total_2
        }

        csv_writer.writerow(info)
        print(x_value, total_1, total_2)

        x_value += 1
        total_1 = total_1 + random.randint(-6, 8)
        total_2 = total_2 + random.randint(-5, 6)

    time.sleep(1)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation

# Function to update the plot in each frame
def update(frame):
    y = np.sin(x + frame * 0.1)
    line.set_ydata(y)

    # Set x-axis limits to keep the scale constant
    ax.set_xlim(x.min(), x.max())

    return line,

# Sample data
x = np.linspace(0, 10000, 100)
y = np.sin(x)

# Create the figure and axis
fig, ax = plt.subplots()
line, = ax.plot(x, y)
ax.set_title('Dynamic Plot')
ax.set_xlabel('X-axis')
ax.set_ylabel('Y-axis')

# Create the animation
animation = FuncAnimation(fig, update, frames=100, interval=100)

# Display the plot
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, clear_output

# Khởi tạo đồ thị
fig, ax = plt.subplots()
x_values = []
y_values = []
line, = ax.plot(x_values, y_values)

# Hàm cập nhật đồ thị
def update_plot(new_data):
    x_values.append(new_data[0])
    y_values.append(new_data[1])
    line.set_xdata(x_values)
    line.set_ydata(y_values)
    ax.relim()
    ax.autoscale_view()
    display(fig)
    clear_output(wait=True)

# Mô phỏng việc nhận được dữ liệu mới và cập nhật đồ thị
for i in range(10):
    new_data_point = [i, np.sin(i)]  # Thay thế bằng dữ liệu thực tế của bạn
    update_plot(new_data_point)


In [ ]:
data = pd.read_csv("strain_groundtruth/strain_1785.csv")
x = data['Sim_step']
y = data['lamda_xz']
fig, ax = plt.subplots()
ax.plot(x,y,label="groundtruth_data")

data1 = pd.read_csv("strain_groundtruth/strain_1392.csv")
x1 = data1['Sim_step']
y1 = data1['lamda_xz']
ax.plot(x1,y1,label="error")
ax.legend(loc='upper left')

current_cost = np.sqrt(np.mean((y[1:] - y1[1:])**2))
print(current_cost)

In [ ]:
import numpy as np
import math as m
from design_space import get_robot_class

check = get_robot_class('Leg')
check


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
categories = np.array([80,90,100]) # Categories from 1 to 10
initial_logits = torch.ones(3,dtype=torch.float)
initial_beta = 1.0  # Initial inverse temperature
# Function to calculate Gibbs distribution
def gibbs_distribution(energy, beta):
    return np.exp(-beta * energy)

# Function to update Gibbs distribution over time
def update_gibbs_distribution(beta, num_steps):
    energies = np.linspace(0, 5, 100)  # Energy levels from 0 to 10
    probabilities = gibbs_distribution(energies, beta)
    plt.plot(energies, probabilities, label='Initial Distribution')

    for _ in range(num_steps):
        beta += 0.2  # Increment beta over time
        new_probabilities = gibbs_distribution(energies, beta)
        plt.plot(energies, new_probabilities, linestyle='--', label=f'Beta = {beta:.1f}')

    plt.title('Gibbs Distribution Over Time')
    plt.xlabel('Energy')
    plt.ylabel('Probability')
    plt.legend()
    plt.show()

# Initial parameters
initial_beta = 0  # Inverse temperature
num_update_steps = 5  # Number of update steps

# Update and plot Gibbs distribution over time
update_gibbs_distribution(initial_beta, num_update_steps)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Function to calculate Gibbs distribution for categories
def gibbs_distribution_for_categories(categories, beta):
    # Calculate unnormalized probabilities using Gibbs distribution formula
    unnormalized_probabilities = np.exp(-beta * categories)
    
    # Normalize probabilities
    normalization_factor = np.sum(unnormalized_probabilities)
    probabilities = unnormalized_probabilities / normalization_factor
    
    return probabilities

# Function to update Gibbs distribution for categories over time
def update_gibbs_distribution_for_categories(categories, initial_beta, num_steps):
    plt.plot(categories, gibbs_distribution_for_categories(categories, initial_beta), label='Initial Distribution')

    for step in range(1, num_steps + 1):
        beta = initial_beta * step  # Increase beta over time
        plt.plot(categories, gibbs_distribution_for_categories(categories, beta), linestyle='--', label=f'Beta = {beta:.1f}')

    plt.title('Gibbs Distribution for Categories Over Time')
    plt.xlabel('Categories')
    plt.ylabel('Probability')
    plt.legend()
    plt.show()

# Initial parameters
categories = np.arange(1, 11)  # Categories from 1 to 10
initial_beta = 1.0  # Initial inverse temperature
num_steps = 5  # Number of update steps

# Update and plot Gibbs distribution for categories over time
update_gibbs_distribution_for_categories(categories, initial_beta, num_steps)


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from design_space.design_space import whiskerdesignspace
import gym
# Custom categorical distribution using dl.CatDist
class CustomCategoricalDistribution:
    def __init__(self, logits):
        self.dist = torch.distributions.Categorical(logits=logits)

    def sample(self):
        return self.dist.sample()

# Function to calculate Gibbs distribution for categories using the custom distribution (CatDist)
def gibbs_distribution_for_categories(categories, logits, beta):
    custom_dist = CustomCategoricalDistribution(logits * beta)
    return custom_dist

# Function to update Gibbs distribution for categories over time and sample
def update_gibbs_distribution_for_categories(categories, initial_logits, initial_beta, num_steps):

    plt.plot(categories, gibbs_distribution_for_categories(categories, initial_logits, initial_beta).dist.probs.numpy(), label='Initial Distribution')

    for step in range(1, num_steps + 1):
        updated_logits = initial_logits + torch.tensor(np.random.rand(initial_logits.shape[0]))  # Update logits with random values
        current_distribution = gibbs_distribution_for_categories(categories, updated_logits, initial_beta)  # Calculate current distribution
        plt.plot(categories, current_distribution.dist.probs.numpy(), linestyle='--', label=f'Step = {step}')
        
        # Sample from the current distribution
        sample = current_distribution.sample()
        print(f"Sample at step {step}: Category {categories[sample]}")

    # plt.title('Gibbs Distribution for Categories Over Time')
    # plt.xlabel('Categories')
    # plt.ylabel('Probability')
    # plt.legend()
    # plt.show()
    return sample

# Example custom distribution function (replace this with your own distribution)
def custom_distribution(categories):
    # Example: Quadratic distribution
    return categories ** 2

# Initial parameters
categories = np.array([80,90,100]) # Categories from 1 to 10
design_space = whiskerdesignspace.design_space()

# initial_logits = torch.tensor(custom_distribution(categories), dtype=torch.float)  # Initial logits
initial_logits = torch.ones(len(design_space),dtype=torch.float)
initial_beta = 1.0  # Initial inverse temperature
num_steps = 5  # Number of update steps

# Update and plot Gibbs distribution for categories over time
sample = update_gibbs_distribution_for_categories(design_space, initial_logits, initial_beta, num_steps)
design_space[sample.item()]

In [ ]:
categories = list(range(1,10))
categories

In [ ]:
import math as m
NLEGS = 8
n = 3+1
designs=[]
for d in range(n ** NLEGS):
    design = []
    for i in range(NLEGS):
        design.append((d // (n ** i)) % n)
    if sum([d > 0 for d in design]) >= 3:
        designs.append(design)

print(design)
print(designs)
m.factorial(3)

In [ ]:
import gym
import math as m

no_chamber = 3
no_regulator = 3
body_length = [100, 90, 80]
class whiskerdesignspace():
    def __init__(self):
        self.pressure_range = gym.spaces.Box(low=0.1, high=1, shape=(1,), dtype='float32')
        self.alldesign = self.design_space()
        # self.body_length = gym.spaces.Box(low=80, high=100, shape=(1,), dtype='float32')
        

    def design_space(self):
        design = []
        for i in range(len(body_length)):
            for j in range(1,no_chamber+1):
                sub_array = [body_length[i],j,0,0,0]
                for k in range (2,j+2):
                    sub_array[k] = self.pressure_range.sample()
                design.append(sub_array)
    
        return design

# Create an instance of WhiskerDesignSpace
whisker_design_space_instance = whiskerdesignspace()

# Call the design_space method
designs = whisker_design_space_instance.design_space()
print(designs)


In [ ]:
import os
import json

# Define the file name
file_name = 'data.json'

# Define the new data to be added
new_data = {
    "name": "Alice",
    "age": 30,
    "city": "New York",
    "hobbies": ["reading", "traveling", "swimming"]
}

# Check if the file exists
if os.path.exists(file_name):
    # If the file exists, read the existing data and append new data
    with open(file_name, 'r') as file:
        try:
            data = json.load(file)
        except json.JSONDecodeError:
            data = []
    data.append(new_data)
else:
    # If the file does not exist, create it and write new data
    data = [new_data]

# Write the updated data back to the file
with open(file_name, 'w') as file:
    json.dump(data, file, indent=4)

print(f"Data written to {file_name}")


In [ ]:
import gym
import math as m
from itertools import product
class whiskerdesignspace():
    def __init__(self):
        self.no_chamber = [1,2,3]
        self.body_length = [100, 80, 60]
        # All possible combinations as tuples.
        self.discrete_combinations = list(product(self.no_chamber, self.body_length))
        self.num_combinations = len(self.discrete_combinations)
        self.pressure_range = gym.spaces.Box(low=0.0001, high=0.001, shape=(1,), dtype='float32')
        self.alldesign = self.design_space()
        # self.body_length = gym.spaces.Box(low=80, high=100, shape=(1,), dtype='float32')
        print(self.discrete_combinations)

    def design_space(self):
        design = []
        for i in range(len(body_length)):
            for j in range(1,no_chamber+1):
                sub_array = [body_length[i],j,0,0,0]
                for k in range (2,j+2):
                    sub_array[k] = self.pressure_range.sample().item()
                design.append(sub_array)
    
        return design

ins = whiskerdesignspace()

design_space = ins.design_space()
design_space

In [ ]:
import gym
import numpy as np
import math as m
from itertools import product
class whiskerdesignspace():
    def __init__(self):
        self.num_samples_per_episode = 2
        self.no_chamber = [1,2,3]
        self.body_length = [100, 80, 60]
        # All possible combinations as tuples.
        self.discrete_combinations = list(product(self.no_chamber, self.body_length))
        self.num_combinations = len(self.discrete_combinations)
         # Initialize uniform distribution over discrete combinations.
        self.discrete_probs = np.ones(self.num_combinations) / self.num_combinations

        # -----------------------------
        # Define continuous design space
        # -----------------------------
        # Three continuous parameters defined by (lower_bound, upper_bound)
        self.continuous_bounds = {
            'pressure1': (0.0, 0.1),
            'pressure2': (0.0, 0.1),
            'pressure3': (0.0, 0.1)
        }
        self.continuous_keys = list(self.continuous_bounds.keys())
        self.num_continuous = len(self.continuous_keys)
        print(self.continuous_keys)
        print(self.continuous_bounds)

        # For each discrete combination, associate a continuous Gaussian distribution.
        self.continuous_distributions = {}  # key: discrete combination tuple, value: {'mean':..., 'std':...}
        for combo in self.discrete_combinations:
            mean = []
            std = []
            for param, (lb, ub) in self.continuous_bounds.items():
                mid = (lb + ub) / 2.0
                mean.append(mid)
                std.append((ub - lb) / 4.0)  # arbitrary initial std: fraction of the range.
            self.continuous_distributions[combo] = {
                'mean': np.array(mean),
                'std': np.array(std)
            }

        print(self.continuous_distributions)

    def _sample_design(self):
        # Sample a discrete combination based on current discrete_probs.
        combo_idx = np.random.choice(len(self.discrete_combinations), p=self.discrete_probs)
        discrete_choice = self.discrete_combinations[combo_idx]
        # Sample continuous parameters from the associated Gaussian.
        cont_params = {}
        dist = self.continuous_distributions[discrete_choice]
        for i, param in enumerate(self.continuous_keys):
            lb, ub = self.continuous_bounds[param]
            val = np.random.normal(dist['mean'][i], dist['std'][i])
            val = np.clip(val, lb, ub)
            cont_params[param] = val
        return {"discrete": discrete_choice, "continuous": cont_params}
    
    def _update_distributions(self):
        # -----------------------------
        # Update discrete distribution using a Gibbs (Boltzmann) update.
        # -----------------------------
        
        elite_count = int(self.num_samples_per_episode * self.elite_fraction)
        rewards = np.array(self.rewards)
        elite_indices = rewards.argsort()[-elite_count:]
        elite_samples = [self.samples[i] for i in elite_indices]
        elite_rewards = [self.rewards[i] for i in elite_indices]
        
        avg_rewards = np.zeros(len(self.discrete_combinations))
        counts = np.zeros(len(self.discrete_combinations))
        for sample, r in zip(elite_samples, elite_rewards):
            combo = sample["discrete"]
            idx = self.discrete_combinations.index(combo)
            avg_rewards[idx] += r
            counts[idx] += 1
        for i in range(len(self.discrete_combinations)):
            if counts[i] > 0:
                avg_rewards[i] /= counts[i]
            else:
                avg_rewards[i] = 0.0
        exp_vals = np.exp(avg_rewards / self.temperature)
        self.discrete_probs = exp_vals / np.sum(exp_vals)
        
        # -----------------------------
        # Update continuous distributions for each discrete combination.
        # -----------------------------
        combo_samples = {combo: [] for combo in self.discrete_combinations}
        for sample in elite_samples:
            combo = sample["discrete"]
            values = [sample["continuous"][param] for param in self.continuous_keys]
            combo_samples[combo].append(values)
        for combo, samples in combo_samples.items():
            if samples:
                data = np.array(samples)
                new_mean = np.mean(data, axis=0)
                new_std = np.std(data, axis=0)
                self.continuous_distributions[combo]["mean"] = new_mean
                self.continuous_distributions[combo]["std"] = np.maximum(new_std, 1e-2)
ins = whiskerdesignspace()

In [ ]:
import torch
import torch.nn.functional as F
from torch.distributions import Distribution, Categorical, Normal, Independent, MixtureSameFamily

class RobotDesignDist(Distribution):
    """
    Joint distribution for robot design parameters.

    A robot design is parameterized by 5 elements:
      - Two discrete parameters (represented as a tuple, e.g. (d1, d2))
      - Three continuous parameters

    The joint probability is modeled as:
        P(robot design) = P(discrete) * P(continuous | discrete)
    
    In this formulation the discrete logits are assumed to be computed externally 
    as: logits = reward * (1 / temperature).

    This class simply outputs the joint distribution.
    Other operations such as sampling or updating are handled in a separate class (e.g., DesignOptimizer).

    Parameters:
      - discrete_logits (torch.Tensor): Tensor of shape (N,), where N is the number of discrete combinations.
      - continuous_means (torch.Tensor): Tensor of shape (N, 3) with the mean for the 3 continuous parameters
           for each discrete combination.
      - continuous_stds (torch.Tensor): Tensor of shape (N, 3) with the standard deviations for the 3 continuous parameters.
      - discrete_values (list): List of length N representing the discrete combinations (e.g., [(1, 'A'), (2, 'B'), ...]).
    """
    def __init__(self, discrete_logits, continuous_means, continuous_stds, discrete_values, validate_args=None):
        self.discrete_logits = discrete_logits
        self.continuous_means = continuous_means
        self.continuous_stds = continuous_stds
        self.discrete_values = discrete_values
        super(RobotDesignDist, self).__init__(validate_args=validate_args)
    
    def get_distribution(self):
        """
        Build and return the joint distribution as a MixtureSameFamily object.
        
        The discrete part is modeled by a Categorical distribution with logits self.discrete_logits.
        The continuous part is modeled as an Independent Normal distribution (with diagonal covariance)
        over 3 dimensions for each discrete combination.
        """
        mixture_dist = Categorical(logits=self.discrete_logits)
        component_dist = Independent(Normal(self.continuous_means, self.continuous_stds), 1)
        joint_dist = MixtureSameFamily(mixture_distribution=mixture_dist,
                                       component_distribution=component_dist)
        return joint_dist
    
    def log_prob(self, value):
        """
        Compute the log probability of a given value under the joint distribution.
        
        Here, value is expected to be a tensor corresponding to the continuous part.
        (Note: The discrete part is implicit in the mixture weights.)
        """
        return self.get_distribution().log_prob(value)
    
    def kl(self, other):
        """
        Compute the KL divergence between this joint distribution and another RobotDesignDist.
        
        The KL divergence is approximated as the sum of:
          KL(p(discrete) || q(discrete)) + Σ_i p(discrete=i) * KL(p(continuous|i) || q(continuous|i))
        Assumes that the ordering of discrete_values is the same for both distributions.
        """
        p_probs = F.softmax(self.discrete_logits, dim=0)
        q_probs = F.softmax(other.discrete_logits, dim=0)
        kl_discrete = torch.sum(p_probs * (torch.log(p_probs + 1e-8) - torch.log(q_probs + 1e-8)))
        
        p_cont = Independent(Normal(self.continuous_means, self.continuous_stds), 1)
        q_cont = Independent(Normal(other.continuous_means, other.continuous_stds), 1)
        kl_components = torch.distributions.kl_divergence(p_cont, q_cont)  # shape: (N,)
        kl_continuous = torch.sum(p_probs * kl_components)
        
        return kl_discrete + kl_continuous
    
    def to_tensors(self):
        """
        Serialize the distribution parameters to a dictionary.
        """
        return {
            'discrete_logits': self.discrete_logits,
            'continuous_means': self.continuous_means,
            'continuous_stds': self.continuous_stds,
            'discrete_values': self.discrete_values
        }
    
    @classmethod
    def from_tensors(cls, tensors):
        """
        Reconstruct a RobotDesignDist instance from a dictionary of tensors.
        """
        return cls(tensors['discrete_logits'], tensors['continuous_means'], 
                   tensors['continuous_stds'], tensors['discrete_values'])


In [ ]:
import torch
import torch.nn.functional as F

# Assume the RobotDesignDist class (as defined previously) is already imported.
# For clarity, here is a brief reminder of the expected inputs:
# - discrete_logits: Tensor of shape (N,) for N discrete combinations.
# - continuous_means: Tensor of shape (N, 3) for the mean of the 3 continuous parameters.
# - continuous_stds: Tensor of shape (N, 3) for the standard deviations.
# - discrete_values: List of N tuples, each representing one combination (e.g., (1, 'A')).

def test_robot_design_dist():
    # Define discrete combinations.
    discrete_values = [(1, 'A'), (2, 'B'), (3, 'C')]
    N = len(discrete_values)
    
    # Define the discrete logits (assumed computed externally as reward*(1/temperature)).
    discrete_logits = torch.tensor([0.5, 0.0, -0.5])  # shape: (3,)
    
    # Define continuous parameters for each discrete combination.
    continuous_means = torch.tensor([[0.0, 0.0, 0.0],
                                     [1.0, 1.0, 1.0],
                                     [2.0, 2.0, 2.0]], dtype=torch.float)
    continuous_stds = torch.tensor([[1.0, 1.0, 1.0],
                                    [0.5, 0.5, 0.5],
                                    [1.5, 1.5, 1.5]], dtype=torch.float)
    
    # Create an instance of RobotDesignDist.
    rdd = RobotDesignDist(discrete_logits, continuous_means, continuous_stds, discrete_values)
    
    # Get the joint distribution (MixtureSameFamily) from the instance.
    joint_dist = rdd.get_distribution()
    
    # Sample 5 samples from the joint distribution.
    samples = joint_dist.sample((5,))
    print("Samples (continuous part) from joint distribution:")
    print(samples)
    
    # Compute log probabilities of these samples using the joint distribution.
    log_probs_joint = joint_dist.log_prob(samples)
    print("\nLog probabilities (from joint distribution):")
    print(log_probs_joint)
    
    # Alternatively, use the log_prob method of the RobotDesignDist instance.
    # Note: Here, the log_prob method expects a tensor corresponding to the continuous part.
    log_probs_rdd = rdd.log_prob(samples)
    print("\nLog probabilities (from rdd.log_prob):")
    print(log_probs_rdd)
    
    # Create a second RobotDesignDist instance with slightly different parameters.
    discrete_logits2 = torch.tensor([0.0, 0.5, -0.2])
    continuous_means2 = torch.tensor([[0.1, -0.1, 0.0],
                                      [1.1, 0.9, 1.0],
                                      [1.9, 2.1, 2.0]], dtype=torch.float)
    continuous_stds2 = torch.tensor([[1.1, 1.0, 0.9],
                                     [0.6, 0.5, 0.4],
                                     [1.4, 1.6, 1.5]], dtype=torch.float)
    rdd2 = RobotDesignDist(discrete_logits2, continuous_means2, continuous_stds2, discrete_values)
    
    # Compute the KL divergence between rdd and rdd2.
    kl_value = rdd.kl(rdd2)
    print("\nEstimated KL divergence (rdd || rdd2):", kl_value.item())

if __name__ == '__main__':
    test_robot_design_dist()


In [ ]:
import numpy as np
import torch
import torch.distributions as dist
from dl import nest

class DesignSampler:
    def __init__(self, use_distribution=True):
        # Simulated design space with a simple uniform sampling method
        self.design_space = self._create_design_space()

        # Create a normal distribution if use_distribution=True, else None
        self.design_dist = self._create_design_distribution() if use_distribution else None

    def _create_design_space(self):
        """Creates a dummy design space with a simple sampling function."""
        class DummyDesignSpace:
            def sample(self):
                return [np.random.uniform(-1, 1) for _ in range(3)]  # 3D sample

        return DummyDesignSpace()

    def _create_design_distribution(self):
        """Creates a dummy PyTorch Gaussian distribution over 3D space."""
        mean = torch.tensor([0.0, 0.0, 0.0])  # Mean at origin
        std = torch.tensor([1.0, 1.0, 1.0])  # Standard deviation of 1
        return dist.Normal(mean, std)  # Independent normal distributions

    def _unnorm(self, sample):
        """Dummy unnormalization function (for demonstration)."""
        return sample * 2  # Example transformation: scale by 2

    def _sample_design(self):
        """The function to sample from either design_space or design_dist."""
        if self.design_dist is None:
            return np.array(self.design_space.sample())  # Sample from design_space
        
        with torch.no_grad():
            a = self.design_dist.sample()
            b = self._unnorm(nest.map_structure(
                lambda x: x.numpy(), a
            ))
            return a, b
            # return self._unnorm(nest.map_structure(
            #     lambda x: x.numpy(), self.design_dist.sample()
            # ))

# Instantiate the sampler
sampler = DesignSampler(use_distribution=True)

# Run the function and print the result
a,b = sampler._sample_design()
print("Sampled Design:", a)
print("Sampled Design:", b)



In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Define means and standard deviations for three Gaussians
init_continuous_means = torch.tensor([[0.0, 0.0, 0.0],
                                      [1.0, 1.0, 1.0],
                                      [2.0, 2.0, 2.0]], dtype=torch.float)

init_continuous_stds = torch.tensor([[1.0, 1.0, 1.0],
                                     [0.5, 0.5, 0.5],
                                     [1.5, 1.5, 1.5]], dtype=torch.float)

# Number of samples per Gaussian
num_samples = 500

# Sample from each Gaussian distribution
samples = []
labels = []

for i in range(init_continuous_means.shape[0]):
    mean = init_continuous_means[i]
    std = init_continuous_stds[i]
    
    dist = torch.distributions.Normal(mean, std)
    sample = dist.sample((num_samples,))  # Generate samples
    samples.append(sample.numpy())  # Convert to NumPy for plotting
    labels.extend([i] * num_samples)  # Keep track of which Gaussian each sample comes from

# Convert to NumPy array for easier handling
samples = np.vstack(samples)
labels = np.array(labels)

# Create 3D scatter plot
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot colored by Gaussian component
ax.scatter(samples[:, 0], samples[:, 1], samples[:, 2], c=labels, cmap='viridis', alpha=0.6)

# Labels and title
ax.set_xlabel("X-axis")
ax.set_ylabel("Y-axis")
ax.set_zlabel("Z-axis")
ax.set_title("3D Gaussian Distributions")

plt.show()


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Select the first Gaussian distribution
mean = torch.tensor([0.0, 0.0, 0.0])
std = torch.tensor([1.0, 1.0, 1.0])

# Create the Normal distribution
dist = torch.distributions.Normal(mean, std)

# Sample data
num_samples = 1000
samples = dist.sample((num_samples,)).numpy()  # Convert to NumPy

# Plot histograms for each dimension
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

dims = ['X', 'Y', 'Z']
for i in range(3):
    axes[i].hist(samples[:, i], bins=30, density=True, alpha=0.6, color='b', edgecolor='black')
    axes[i].set_title(f"Histogram of {dims[i]}")
    axes[i].set_xlabel(f"{dims[i]} values")
    axes[i].set_ylabel("Density")

plt.tight_layout()
plt.show()


In [ ]:
no_chamber = [1,2,3]
no_regulator = 3
pressure_range = torch.tensor([0,0.01])
body_length = [100, 80, 60]
body_parameters = list(product(body_length, no_chamber))
# body parameter is a tensor including body length (being cut) and number chamber in tuple format
N = len(body_parameters)
D = len(no_chamber)
# Initialize discrete scores (which, when multiplied by beta, form logits).
scores = torch.nn.Parameter(torch.zeros((N,), dtype=torch.float32), requires_grad=False)
pressure_range.mean()
# continuous_means = torch.full((N , D), np.mean(pressure_range).item())
# continuous_stds = torch.full((N , D), np.mean(pressure_range).item()/2)

In [ ]:
a = {'body_length': torch.tensor([60,3],dtype=float)}
a['body_length'][0].numpy().item()


In [ ]:
from itertools import product
no_chamber = torch.tensor([1,2,3])
pressure_range = torch.tensor([0,0.01])
body_length = torch.tensor([100, 80, 60])
source = list(product(body_length,no_chamber))
design = {'body_length': 60, 'no_chamber': 2, 'pressure1': 0.03, 'pressure2': 0.04, 'pressure3': 0.02}
# a = product(torch.tensor([design['body_length']]),torch.tensor([design['no_chamber']]))
a = (design['body_length'],design['no_chamber'])
# source
source.index(a)
np.log(10)

In [ ]:
from design_space import design_space
no_chamber = torch.tensor([1,2,3])
pressure_range = torch.tensor([0,0.001])
body_length = torch.tensor([100, 80, 60])
ins = design_space.whiskerdesignspace(body_length,no_chamber,pressure_range)
space = ins.design_space()
first_key = list(space.keys())[0]  # Get the first key (tensor)

first_sub_dict = space[first_key]
for idx,val in first_sub_dict.items():
    print(idx)
    print(val.tolist())

for idx,val in enumerate(first_sub_dict.items()):
    print(idx)
    print(val[0])
    print(val[1].tolist())
print(list(enumerate(first_sub_dict.items())))

In [ ]:
from collections import deque
no_chamber = torch.tensor([1,2,3])
DESIGNS_100 =  [{'body_length': 100, 'no_chamber': 2, 'pressure1': 0.000500157184433192, 'pressure2': 0.0005001485114917159, 'pressure3': 0.0005001524114049971}, 
                           {'body_length': 100, 'no_chamber': 1, 'pressure1': 0.0005000722594559193, 'pressure2': 0.0005000594537705183, 'pressure3': 0.0005001836689189076}, 
                           {'body_length': 100, 'no_chamber': 3, 'pressure1': 0.0005000497912988067, 'pressure2': 0.000500065681990236, 'pressure3': 0.0005001485696993768}, 
                           {'body_length': 100, 'no_chamber': 3, 'pressure1': 0.0005000109667889774, 'pressure2': 0.0005001624231226742, 'pressure3': 0.0005001963581889868}]
REWARDS_100 =  [np.float64(5.8020299007921494e-05), np.float64(9.052226209860237e-05), np.float64(5.827895449783682e-05), np.float64(6.686179029102846e-05)]

for sample, reward in zip(DESIGNS_100, REWARDS_100):
    d_val = sample['no_chamber']
    idx = no_chamber.tolist().index(d_val)
    print(idx)

In [ ]:
import torch
import torch
import torch.nn as nn
import torch.nn.functional as F
class RobotDesignDist(nn.Module):
    """
    Joint distribution for robot design parameters.
    
    A robot design is parameterized by 5 elements:
      - Two discrete parameters (represented as a tuple, e.g. (d1, d2))
      - Three continuous parameters.
    
    The joint probability is modeled as:
    
         P(robot design) = P(discrete) * P(continuous | discrete)
    
    Inputs:
      - discrete_logits: Tensor of shape (N,), where N is the number of discrete combinations.
      - continuous_means: Tensor of shape (N, 3) giving the mean for the 3 continuous parameters per combination.
      - continuous_stds: Tensor of shape (N, 3) giving the std for the 3 continuous parameters per combination.
      - discrete_values: List of length N of tuples (each tuple is one discrete combination).
    """
    def __init__(self):
        super().__init__()
        self.discrete_logits = 0
        self.continuous_means = torch.full((110 , 2), 0.1)  # or init mean = self.pressure_range.mean().item()
        self.continuous_stds = torch.full((110 , 2), 0.05) 

    def get_distribution(self):
        
        self.base_normal = torch.distributions.Normal(self.continuous_means, self.continuous_stds)
        self.transforms = [torch.distributions.SigmoidTransform(), 
                            torch.distributions.AffineTransform(loc=0, scale=0.2)]
        self.transformed_cont_dist = torch.distributions.TransformedDistribution(self.base_normal, self.transforms)

ins = RobotDesignDist()
a = ins.get_distribution()
sample_transformed=ins.transformed_cont_dist.sample()
# sample_transformed = transformed_cont_dist.sample()


# Starting with the sample from the transformed distribution:
sample_original = sample_transformed
# Loop over the transforms in reverse order and apply the inverse:
for transform in reversed(ins.transformed_cont_dist.transforms):
    sample_original = transform.inv(sample_original)

# print("Transformed sample:", sample_transformed[0].item()[0])
print("Recovered original sample:", sample_original[0][0].item())


In [ ]:
import yaml
from pathlib import Path

class MyRLAgent:
    def __init__(self, env_id, algo_name, seed=0, output_dir="./Results", **kwargs):
        self.env_id = env_id
        self.algo_name = algo_name
        self.seed = seed
        self.output_dir = output_dir
        
        # # Retrieve optional parameters from kwargs
        # self.learning_rate = kwargs.get("learning_rate", 0.001)
        # self.gamma = kwargs.get("gamma", 0.99)
        self.extra_options = kwargs  # You can also store the entire dict if needed.
        # print("Initialized MyRLAgent with:")
        # print(f"  env_id: {env_id}")
        # print(f"  algo_name: {algo_name}")
        # print(f"  seed: {seed}")
        # print(f"  output_dir: {output_dir}")
        # print(f"  learning_rate: {self.learning_rate}")
        # print(f"  gamma: {self.gamma}")
        # print(f"  extra options: {self.extra_options}")
        self.load()
    def load(self):
        # Retrieve optional parameters from kwargs
        self.learning_rate = self.extra_options["learning_rate"]
        self.gamma = self.extra_options["gamma"]
        print("Initialized MyRLAgent with:")
        print(f"  env_id: {self.env_id}")
        print(f"  algo_name: {self.algo_name}")
        print(f"  seed: {self.seed}")
        print(f"  output_dir: {self.output_dir}")
        print(f"  learning_rate: {self.learning_rate}")
        print(f"  gamma: {self.gamma}")
        print(f"  extra options: {self.extra_options}")
        
        self.extra_options = None
        print(f"  extra options: {self.extra_options}")

        if not self.extra_options:
            self.params_path = "/home/nhnhan/Desktop/sofa/SofaGym/agents/hyperparameters/stable_baselines_params.yml"
            config = yaml.safe_load(Path(self.params_path).read_text())
            self.params = config[self.algo_name]
            print("updated self.params : ", self.params)


# Creating an instance:
agent = MyRLAgent("CartPole-v1", "PPO", learning_rate=0.0005, gamma=0.95, custom_flag=True)


In [ ]:
from itertools import product
import torch
import numpy as np
a = [1, 3, 5]
b = [2, 4]

test = list(product(a, b))        # test is a list of tuples, e.g. [(1, 2), (1, 4), ...]
test_tensor = torch.tensor(test)  # Convert to a 2D tensor
print(test_tensor)
print(test)


a = {'1': 1,
     '2': 2}

ze = np.zeros(len(test))
ze

In [4]:
test = {
    '100': {'1': 1, '2': 2, '3': 3},
    '90':  {'1': 4, '2': 5, '3': 6},
    '80':  {'1': 7, '2': 8, '3': 9},
    '70':  {'1': 10, '2': 11, '3': 12}
}

# Step 1: Sum the values for each inner key across all body_length categories.
sums = {}
for body_length, inner in test.items():
    for key, value in inner.items():
        # Here, key is the "second item" (e.g., '1', '2', '3').
        sums[key] = sums.get(key, 0) + value

print("Sums:", sums)  # Expected: {'1': 4, '2': 8, '3': 12}

# Step 2: Determine the maximum sum.
max_sum = max(sums.values())
min_sum = min(sums.values())
rw_range = max_sum - min_sum
# Step 3: Compute the extra reward for each inner key as the ratio to max_sum.
extra_rewards = {k: (v - min_sum) / rw_range for k, v in sums.items()}

print("Extra Rewards:", extra_rewards)

test2 = {}

for i in range(3):
    test2[i] = 0

test2

Sums: {'1': 22, '2': 26, '3': 30}
Extra Rewards: {'1': 0.0, '2': 0.5, '3': 1.0}


{0: 0, 1: 0, 2: 0}